## Objetivo del cuaderno

Este cuaderno carga datos de Idealista, los limpia y finalmente enriquece el dataset con distancias minimas a POIs usando `geospatial_expansion`.

In [10]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# ConfiguraciÃ³n para encontrar la raÃ­z del proyecto y agregarla al sys.path
# Necesario para importar mÃ³dulos desde 'src'
cwd = Path.cwd().resolve()
project_root = next((p for p in [cwd, *cwd.parents] if (p / "src").exists()), None)
if project_root is None:
    raise RuntimeError("No se encontro la raiz del proyecto (carpeta 'src').")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

#### Import del modulo geoespacial

Se importa en una celda separada para aislar errores de entorno/ruta del resto de la configuracion.

In [11]:
from src.geospatial_expansion import agregar_distancias_minimas_poi

## Carga de datos

Se lee el CSV procesado de Idealista y se revisan las primeras filas.

In [12]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

DATA_DIR = Path("../../data/raw/idealistaAPI/preprocess/rent_homes_run_20260220_111903")
DATA_IDEALISTA = DATA_DIR / "rent_homes_cantabria_bezana_like_raw.csv"

df = pd.read_csv(DATA_IDEALISTA)

df.head()

,propertyCode,thumbnail,numPhotos,price,propertyType,operation,size,rooms,bathrooms,address,province,municipality,district,country,latitude,longitude,showAddress,url,distance,description,hasVideo,status,newDevelopment,hasLift,priceByArea,hasPlan,has3DTour,has360,hasStaging,notes,topNewDevelopment,newDevelopmentHighlight,topPlus,priceInfo.price.amount,priceInfo.price.currencySuffix,detailedType.typology,detailedType.subTypology,suggestedTexts.subtitle,suggestedTexts.title,externalReference,floor,exterior,parkingSpace.hasParkingSpace,parkingSpace.isParkingSpaceIncludedInPrice,priceInfo.price.priceDropInfo.formerPrice,priceInfo.price.priceDropInfo.priceDropValue,priceInfo.price.priceDropInfo.priceDropPercentage,parkingSpace.parkingSpacePrice
0,110695153,https://img4.idealista.com/blur/480_360_mq/0/i...,13,800.0,studio,rent,40.0,0,1,calle la Universidad,Cantabria,Santander,Los Castros,es,43.470612,-3.795827,False,https://www.idealista.com/inmueble/110695153/,9206,Este estudio completamente amueblado se alquil...,False,good,False,False,20.0,False,False,False,False,[],False,False,False,800.0,€/mes,flat,studio,"Los Castros, Santander",Estudio en calle la Universidad,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,110690044,https://img4.idealista.com/blur/480_360_mq/0/i...,16,890.0,flat,rent,82.0,2,1,BO POZO (EL),Cantabria,Piélagos,Boo,es,43.432578,-3.946268,False,https://www.idealista.com/inmueble/110690044/,3652,Descubre este encantador BAJO CON JARDIN ubica...,False,good,False,False,11.0,False,False,False,False,[],False,False,False,890.0,€/mes,flat,NaN,"Boo, Piélagos",Piso en Bo Pozo (el),ACR-01952,bj,True,True,True,NaN,NaN,NaN,NaN
2,99811129,https://img4.idealista.com/blur/480_360_mq/0/i...,35,890.0,flat,rent,72.0,3,1,calle los Caños,Cantabria,Santander,General Dávila,es,43.465622,-3.812111,False,https://www.idealista.com/inmueble/99811129/,7783,Housfy alquila � piso en Santander. La propied...,True,good,False,False,12.0,False,False,False,False,[],False,False,False,890.0,€/mes,flat,NaN,"General Dávila, Santander",Piso en calle los Caños,645047,5,True,True,True,NaN,NaN,NaN,NaN
3,110689824,https://img4.idealista.com/blur/480_360_mq/0/i...,9,780.0,flat,rent,64.0,1,1,"calle San Fernando, 16",Cantabria,Santander,Numancia - San Fernando,es,43.460766,-3.819301,True,https://www.idealista.com/inmueble/110689824/,7070,ALQUILER FIJO Vivienda con estupenda situació...,False,good,False,True,12.0,False,False,False,False,[],False,False,False,780.0,€/mes,flat,NaN,"Numancia - San Fernando, Santander","Piso en calle San Fernando, 16",SA1528,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,110687546,https://img4.idealista.com/blur/480_360_mq/0/i...,54,1400.0,flat,rent,132.0,3,2,calle Navas de Tolosa,Cantabria,Santander,Centro - Ayuntamiento,es,43.459235,-3.807487,False,https://www.idealista.com/inmueble/110687546/,7952,Se ofrece en alquiler magnífico piso situado e...,False,good,False,True,11.0,False,False,False,False,[],False,False,False,1400.0,€/mes,flat,NaN,"Centro - Ayuntamiento, Santander",Piso en calle Navas de Tolosa,22419,1,True,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
print(f"Columnas totales: {df.columns}")

columns = ['propertyCode','price','size','rooms','bathrooms','detailedType.typology','detailedType.subTypology',
                'province','municipality','district','latitude','longitude','floor','exterior','hasLift',
                'parkingSpace.hasParkingSpace','newDevelopment']

df_reduced = df[columns].copy()
df_reduced.head()

Columnas totales: Index(['propertyCode', 'thumbnail', 'numPhotos', 'price', 'propertyType', 'operation', 'size', 'rooms', 'bathrooms', 'address', 'province',
       'municipality', 'district', 'country', 'latitude', 'longitude', 'showAddress', 'url', 'distance', 'description', 'hasVideo',
       'status', 'newDevelopment', 'hasLift', 'priceByArea', 'hasPlan', 'has3DTour', 'has360', 'hasStaging', 'notes', 'topNewDevelopment',
       'newDevelopmentHighlight', 'topPlus', 'priceInfo.price.amount', 'priceInfo.price.currencySuffix', 'detailedType.typology',
       'detailedType.subTypology', 'suggestedTexts.subtitle', 'suggestedTexts.title', 'externalReference', 'floor', 'exterior',
       'parkingSpace.hasParkingSpace', 'parkingSpace.isParkingSpaceIncludedInPrice', 'priceInfo.price.priceDropInfo.formerPrice',
       'priceInfo.price.priceDropInfo.priceDropValue', 'priceInfo.price.priceDropInfo.priceDropPercentage',
       'parkingSpace.parkingSpacePrice'],
      dtype='str')


,propertyCode,price,size,rooms,bathrooms,detailedType.typology,detailedType.subTypology,province,municipality,district,latitude,longitude,floor,exterior,hasLift,parkingSpace.hasParkingSpace,newDevelopment
0,110695153,800.0,40.0,0,1,flat,studio,Cantabria,Santander,Los Castros,43.470612,-3.795827,NaN,NaN,False,NaN,False
1,110690044,890.0,82.0,2,1,flat,NaN,Cantabria,Piélagos,Boo,43.432578,-3.946268,bj,True,False,True,False
2,99811129,890.0,72.0,3,1,flat,NaN,Cantabria,Santander,General Dávila,43.465622,-3.812111,5,True,False,True,False
3,110689824,780.0,64.0,1,1,flat,NaN,Cantabria,Santander,Numancia - San Fernando,43.460766,-3.819301,3,NaN,True,NaN,False
4,110687546,1400.0,132.0,3,2,flat,NaN,Cantabria,Santander,Centro - Ayuntamiento,43.459235,-3.807487,1,True,True,NaN,False


In [14]:
columnas_espanol = {
    "propertyCode": "codigo_inmueble",
    "price": "precio",
    "size": "superficie_construida_m2",
    "rooms": "numero_dormitorios",
    "bathrooms": "numero_banos",
    "detailedType.typology": "tipologia",
    "detailedType.subTypology": "subtipologia",
    "operation": "tipo_operacion",
    "province": "provincia",
    "municipality": "municipio",
    "district": "distrito",
    "latitude": "latitud",
    "longitude": "longitud",
    "floor": "planta",
    "exterior": "es_exterior",
    "hasLift": "tiene_ascensor",
    "parkingSpace.hasParkingSpace": "tiene_garaje",
    "newDevelopment": "obra_nueva"
}

df_es = df_reduced.rename(columns = columnas_espanol).copy()
df_es.head()

,codigo_inmueble,precio,superficie_construida_m2,numero_dormitorios,numero_banos,tipologia,subtipologia,provincia,municipio,distrito,latitud,longitud,planta,es_exterior,tiene_ascensor,tiene_garaje,obra_nueva
0,110695153,800.0,40.0,0,1,flat,studio,Cantabria,Santander,Los Castros,43.470612,-3.795827,NaN,NaN,False,NaN,False
1,110690044,890.0,82.0,2,1,flat,NaN,Cantabria,Piélagos,Boo,43.432578,-3.946268,bj,True,False,True,False
2,99811129,890.0,72.0,3,1,flat,NaN,Cantabria,Santander,General Dávila,43.465622,-3.812111,5,True,False,True,False
3,110689824,780.0,64.0,1,1,flat,NaN,Cantabria,Santander,Numancia - San Fernando,43.460766,-3.819301,3,NaN,True,NaN,False
4,110687546,1400.0,132.0,3,2,flat,NaN,Cantabria,Santander,Centro - Ayuntamiento,43.459235,-3.807487,1,True,True,NaN,False


In [15]:
print(f"Total elementos duplicados: {df_es['codigo_inmueble'].duplicated().sum()} de {len(df_es)}")
df_es[df_es['codigo_inmueble'].duplicated(keep=False)].sort_values("codigo_inmueble").head()


Total elementos duplicados: 1872 de 2365


,codigo_inmueble,precio,superficie_construida_m2,numero_dormitorios,numero_banos,tipologia,subtipologia,provincia,municipio,distrito,latitud,longitud,planta,es_exterior,tiene_ascensor,tiene_garaje,obra_nueva
2146,30232461,1500.0,130.0,4,3,flat,NaN,Cantabria,Santander,El Sardinero,43.472191,-3.787613,1,NaN,True,NaN,False
2305,30232461,1500.0,130.0,4,3,flat,NaN,Cantabria,Santander,El Sardinero,43.472191,-3.787613,1,NaN,True,NaN,False
2354,30232461,1500.0,130.0,4,3,flat,NaN,Cantabria,Santander,El Sardinero,43.472191,-3.787613,1,NaN,True,NaN,False
2273,30232461,1500.0,130.0,4,3,flat,NaN,Cantabria,Santander,El Sardinero,43.472191,-3.787613,1,NaN,True,NaN,False
2227,30232461,1500.0,130.0,4,3,flat,NaN,Cantabria,Santander,El Sardinero,43.472191,-3.787613,1,NaN,True,NaN,False


In [16]:
df_clean = df_es.drop_duplicates(subset="codigo_inmueble", keep="first").copy()
print(f"Forma del nuevo dataset: {df_clean.shape}")


Forma del nuevo dataset: (493, 17)


In [17]:
df_clean['municipio'].value_counts().head(10)

municipio
Santander               284
Laredo                   40
Castro-Urdiales          33
Piélagos                 13
Santa Cruz de Bezana     13
Camargo                  12
Torrelavega              12
Suances                  11
El Astillero             11
Voto                      6
Name: count, dtype: int64

In [18]:
df_expandido = agregar_distancias_minimas_poi(df_clean, ["playa", "supermercado", "colegio"])

df_expandido.head()

,codigo_inmueble,precio,superficie_construida_m2,numero_dormitorios,numero_banos,tipologia,subtipologia,provincia,municipio,distrito,latitud,longitud,planta,es_exterior,tiene_ascensor,tiene_garaje,obra_nueva,distancia_min_playa_km,distancia_min_supermercado_km,distancia_min_colegio_km
0,110695153,800.0,40.0,0,1,flat,studio,Cantabria,Santander,Los Castros,43.470612,-3.795827,NaN,NaN,False,NaN,False,1.023,0.259,0.351
1,110690044,890.0,82.0,2,1,flat,NaN,Cantabria,Piélagos,Boo,43.432578,-3.946268,bj,True,False,True,False,2.158,2.107,0.998
2,99811129,890.0,72.0,3,1,flat,NaN,Cantabria,Santander,General Dávila,43.465622,-3.812111,5,True,False,True,False,2.158,0.088,0.073
3,110689824,780.0,64.0,1,1,flat,NaN,Cantabria,Santander,Numancia - San Fernando,43.460766,-3.819301,3,NaN,True,NaN,False,2.515,0.094,0.181
4,110687546,1400.0,132.0,3,2,flat,NaN,Cantabria,Santander,Centro - Ayuntamiento,43.459235,-3.807487,1,True,True,NaN,False,1.804,0.187,0.310
